(z3-knight)=
# 骑士巡游：找一条把每个格子都走到、只走一次的闭合路线

## 这一章讲什么

让国际象棋里的骑士在 8×8 的棋盘上走出一条路线，把每个格子恰好访问一次，走完还要回得到起点，收成一条闭合路线。由于每一步都受骑士走法的限制，因此这条路线并不是随便排一排就能得到的。

- 每个格子给一个整数，表示第几个访问这里
- 相邻的两个编号必须落在骑士一步能到的地方，这一句用 `PbEq()` 写
- 规则照原样变成约束，Z3 几秒之内就交出路线

骑士巡游问题在 [用 OrTools 求解骑士巡游问题](ortools-knight) 里也讲过，两章的模型可以对照着看。

## 思路

给每个格子分配一个 `0` 到 `63` 的整数，让这个数就代表第几个访问它，访问的先后顺序便藏在这一组数里，后面所有约束都围绕它展开。

- 所有格子的值都落在 0～63 之间，而且互不相同(`Distinct`)
- 值为 `c` 的格子旁边，一定有个值为 `c+1` 的格子
- 值为 `c` 的格子旁边，也一定有个值为 `c-1` 的格子
- 最后那个(第 63 个)格子旁边，一定站着第 0 个格子，路线才闭得上

这里的“旁边”只算骑士那八种 L 字形走法能到的格子。

由于约束只认骑士能走的范围，因此路线顺着这个范围自己定了下来。

In [1]:
from z3 import *
from itertools import product

# 骑士的 8 个走法方向
KNIGHT_MOVES = [(1, 2), (2, 1), (-1, 2), (2, -1), (-2, 1), (1, -2), (-1, -2), (-2, -1)]

## 代码

棋盘上每个格子建一个整数变量，再把约束一条一条加进去。

In [2]:
w, h = 8, 8
N = w * h

cells = {}
solver = Solver()

for i, j in product(range(h), range(w)):
    c = cells[i, j] = Int(f'c_{i}_{j}')
    solver.add(c >= 0, c <= N - 1)

# 访问顺序互不相同
solver.add(Distinct(list(cells.values())))

for i, j in product(range(h), range(w)):
    c = cells[i, j]
    neighbours = [cells[i + di, j + dj] for di, dj in KNIGHT_MOVES
                  if (i + di, j + dj) in cells]

    next_is = [(x == c + 1, 1) for x in neighbours]   # 周围是否有 c+1
    prev_is = [(x == c - 1, 1) for x in neighbours]   # 周围是否有 c-1
    start_is = [(x == 0, 1) for x in neighbours]      # 周围是否有 0

    # 除最后一个格子（63）外，周围必须有一个 c+1
    solver.add(Implies(c < N - 1, PbEq(next_is, 1)))
    # 除第一个格子（0）外，周围必须有一个 c-1
    solver.add(Implies(c > 0, PbEq(prev_is, 1)))
    # 最后一个格子（63）的周围必须有 0（闭合路线）
    solver.add(Implies(c == N - 1, PbEq(start_is, 1)))

%time solver.check()
m = solver.model()

for i in range(h):
    for j in range(w):
        print(f'{m[cells[i, j]].as_long():2d}', end=' ')
    print()

CPU times: total: 4 s
Wall time: 5.02 s
63 60 45 10 47 58 23 20 
44 11 62 59 24 21 48 57 
61  0  9 46 27 56 19 22 
12 43 36  3 54 25 28 49 
37  8  1 26 35  4 55 18 
42 13 40 53  2 31 50 29 
 7 38 15 34  5 52 17 32 
14 41  6 39 16 33 30 51 


表里每个格子都填上了第几个访问，请读者核对 0 到 63 是不是各出现了一次。

## 验算找到的路线

Z3 说是骑士巡游，未必真的满足条件，因此再写一段自己检查的代码，心里才踏实。

In [3]:
def check_tour(board):
    """按骑士的移动顺序，逐个确认每个格子的值"""
    h, w = len(board), len(board[0])
    pos = {board[i][j]: (i, j) for i in range(h) for j in range(w)}

    if sorted(pos) != list(range(h * w)):
        return "编号没有做到 0〜63 各一次"

    for k in range(h * w):
        i, j = pos[k]
        i2, j2 = pos[(k + 1) % (h * w)]
        if (i2 - i, j2 - j) not in KNIGHT_MOVES:
            return f"{k} 号格到 {k + 1} 号格的移动不是骑士的走法"

    return "OK: 按骑士走法 0 → 1 → … → 63 → 0 走完一圈"


tour = [[m[cells[i, j]].as_long() for j in range(w)] for i in range(h)]
print(check_tour(tour))

OK: 按骑士走法 0 → 1 → … → 63 → 0 走完一圈


## 约束的含义

| 约束 | 含义 |
|---|---|
| `Implies(c < N - 1, PbEq(next_is, 1))` | 除最后一个外，周围必须有一个 `c+1` 的格子 |
| `Implies(c > 0, PbEq(prev_is, 1))` | 除第一个外，周围必须有一个 `c-1` 的格子 |
| `Implies(c == N - 1, PbEq(start_is, 1))` | 最后一个格子的周围必须有一个 `0` 的格子 |
| `Distinct(...)` | 0～63 不重复 |

“周围有个 `c+1`”与“周围有个 `c-1`”，只写一边即可，因为站在 `c+1` 那个格子往回看，`c` 就是它的 `c-1`。注意，两边都写上，是让 Z3 搜起来更省力。

## 什么是 `PbEq()`

`PbEq(args, k)` 管的是加权和恰好等于 `k`：`args` 里放一串 `(条件, 权重)`，哪个条件为 `True`，它的权重就记进总和。

例如权重 1 的条件有 2 个为真，权重 2 的有 1 个为真，加起来就是 4。

In [4]:
x, y, z = Bools('x y z')

s = Solver()
s.add(PbEq([(x, 1), (y, 2), (z, 3)], 5))
s.check()
print(s.model())

[y = True, z = True, x = False]


`x` 记 1 分、`y` 记 2 分、`z` 记 3 分，要让总分正好是 5，选中的就是 `y` 和 `z`，这个简单的例子说明了它的计算方式。

骑士巡游的代码里，每个条件的权重都写成 1，这时候 `PbEq(..., 1)` 说的就是为真的条件恰好一个。

同样的含义，用 `Sum(...) == 1` 也写得出来。可是 `PbEq()` 把“恰好一个”这层意图摆在明面上，Z3 解起来还快些。

## 小结

- 把访问顺序写成每个格子上的整数，相邻两格必须是骑士一步的距离，路线就定了
- `Distinct` 保证 0～63 每个数只用一次
- “恰好一个”用 `PbEq([...], 1)` 写
- 答案出来以后，自己再写一段验算的代码更稳妥(`check_tour()`)